# 🎙️ ZeroTTS Studio (TSS) - Google Colab Free Tier
**Mô hình Text-to-Speech (TTS) Tiếng Việt với toàn bộ tính năng: Phân đoạn tag `[Câu 1]`, `#[Bỏ qua]`, `[pause: 1.5s]`, gộp `_FULL_MERGED.mp3`, Streaming Audio.**

🔗 Mã nguồn GitHub: [RevenantKitana/TSS](https://github.com/RevenantKitana/TSS)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

---

### ⚡ Hướng dẫn cài đặt nhanh (1-Click Run):
1. **Bật GPU T4 (Khuyên dùng)**: Vào menu **Runtime (Thời gian chạy)** -> **Change runtime type (Thay đổi loại thời gian chạy)** -> Chọn **T4 GPU** -> Nhấn **Save (Lưu)**.
2. **Chạy tuần tự các bước**: 
   - **Bước 1**: Kết nối Google Drive (để lưu file audio vĩnh viễn).
   - **Bước 2**: Tự động nạp mã nguồn từ GitHub [RevenantKitana/TSS](https://github.com/RevenantKitana/TSS).
   - **Bước 3**: Cài đặt thư viện hệ thống và ONNX Runtime GPU (CUDA).
   - **Bước 4**: Tải Model Weights từ Hugging Face.
   - **Bước 5**: Bấm vào link **Public URL** (`https://xxxx.trycloudflare.com`) để mở WebUI Studio!

## ⚙️ Bước 1: Kiểm tra GPU & Kết nối Google Drive
> *Google Drive giúp tự động lưu toàn bộ file âm thanh đã xuất (`outputs/`) vĩnh viễn.*

In [ ]:
#@title Cấu hình Lưu Trữ & Kiểm tra Phần Cứng { run: "auto", display-mode: "form" }
MOUNT_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/ZeroTTS_Outputs"

import os
import subprocess

# 1. Kiem tra GPU
print("🔍 Đang kiểm tra phần cứng...")
try:
    gpu_info = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]).decode("utf-8").strip()
    print(f"✅ Phát hiện GPU: {gpu_info}")
    HAS_GPU = True
except Exception:
    print("ℹ️ Đang chạy trên CPU. Bạn có thể bật GPU tại: Runtime -> Change runtime type -> T4 GPU.")
    HAS_GPU = False

# 2. Mount Google Drive nếu được bật
if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
        os.environ["ZEROTTS_OUTPUT_DIR"] = DRIVE_OUTPUT_DIR
        print(f"📁 File âm thanh sẽ được lưu vĩnh viễn tại Google Drive:")
        print(f"   -> {DRIVE_OUTPUT_DIR}")
    except Exception as e:
        print(f"⚠️ Không thể kết nối Google Drive ({e}). Sử dụng thư mục tạm /content/outputs.")
        os.environ["ZEROTTS_OUTPUT_DIR"] = "/content/outputs"
else:
    os.environ["ZEROTTS_OUTPUT_DIR"] = "/content/outputs"
    print("📁 Dữ liệu sẽ lưu tạm tại /content/outputs (sẽ bị xoá khi tắt Colab).")

## 📦 Bước 2: Nạp Mã Nguồn Trực Tiếp Từ GitHub
> *Tự động clone phiên bản mới nhất từ [https://github.com/RevenantKitana/TSS](https://github.com/RevenantKitana/TSS) — Không cần upload file zip.*

In [ ]:
#@title Nạp Mã Nguồn Từ GitHub (RevenantKitana/TSS)
import os
import sys
import subprocess
import shutil

REPO_URL = "https://github.com/RevenantKitana/TSS.git"
APP_DIR = "/content/TSS"

print(f"📥 Đang nạp mã nguồn từ GitHub: {REPO_URL}...")
if not os.path.exists(os.path.join(APP_DIR, ".git")):
    shutil.rmtree(APP_DIR, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, APP_DIR], check=True)
    print("✅ Clone mã nguồn thành công!")
else:
    print("🔄 Cập nhật mã nguồn mới nhất từ GitHub...")
    subprocess.run(["git", "-C", APP_DIR, "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", APP_DIR, "reset", "--hard", "origin/main"], check=True)
    print("✅ Đã đồng bộ mã nguồn mới nhất!")

# Copy đồng bộ vào /content/webui và /content/src
for folder in ["webui", "src"]:
    src_f = os.path.join(APP_DIR, folder)
    dst_f = os.path.join("/content", folder)
    if os.path.exists(src_f) and not os.path.exists(dst_f):
        try:
            shutil.copytree(src_f, dst_f, dirs_exist_ok=True)
        except Exception:
            pass

if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)
if "/content" not in sys.path:
    sys.path.insert(0, "/content")

%cd $APP_DIR

## 🔧 Bước 3: Cài Đặt Thư Viện Hệ Thống & ONNX Runtime GPU
> *Cài đặt ffmpeg, libportaudio2 và các thư viện cần thiết.*

In [ ]:
#@title Cài Đặt Thư Viện Hệ Thống & Python
print("🔧 Đang cài đặt thư viện hệ thống và Python (chỉ mất ~1 phút)...")
!apt-get update -qq && apt-get install -y -qq ffmpeg libportaudio2

if HAS_GPU:
    print("⚡ Đang cài đặt ONNX Runtime GPU (Tăng tốc CUDA T4)...")
    !pip install -q "onnxruntime-gpu>=1.17.0"
else:
    print("⚙️ Đang cài đặt ONNX Runtime CPU...")
    !pip install -q "onnxruntime>=1.17.0"

!pip install -q soundfile sounddevice fastapi uvicorn tokenizers huggingface_hub scipy requests pydantic
!pip install -q -e .

print("\n✅ Cài đặt hoàn tất! Toàn bộ tính năng (Tag timeline, gộp MP3, WebUI) đã sẵn sàng.")

## 🧠 Bước 4: Tải Model Weights (Mô Hình Pre-trained)

In [ ]:
#@title Tải Model zeroweight-ai/ZeroTTS từ Hugging Face
from huggingface_hub import snapshot_download
import os

APP_DIR = "/content/TSS"
MODEL_DIR = os.path.join(APP_DIR, "ZeroTTS_model") if os.path.exists(APP_DIR) else "/content/ZeroTTS_model"
os.environ["ZEROTTS_MODEL"] = MODEL_DIR

if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    print("⏳ Đang tải mô hình ZeroTTS từ Hugging Face (~500MB)... Vui lòng chờ 10-30 giây.")
    snapshot_download(repo_id="zeroweight-ai/ZeroTTS", local_dir=MODEL_DIR)
    print("✅ Đã tải xong Model Weights!")
else:
    print("✅ Mô hình đã có sẵn trong ZeroTTS_model!")

## 🚀 Bước 5: Khởi Chạy WebUI với Cloudflare Tunnel (Miễn phí 100%)
> *Cloudflare Tunnel sẽ tạo một đường link Public HTTPS an toàn (`https://xxxx.trycloudflare.com`) để bạn truy cập WebUI từ bất kỳ thiết bị nào.*

In [ ]:
#@title Khởi chạy WebUI Studio & Mở Public URL { display-mode: "form" }
import subprocess
import time
import re
import os
import sys
import glob
from IPython.display import display, HTML

# 1. Tự động xác định chính xác vị trí webui/server.py và project root
APP_DIR = "/content/TSS"
server_candidates = [
    os.path.join(APP_DIR, "webui", "server.py"),
    "/content/TSS/webui/server.py",
    "/content/webui/server.py",
    os.path.join(os.getcwd(), "webui", "server.py"),
]
SERVER_SCRIPT = None
PROJECT_ROOT = None
for cand in server_candidates:
    if os.path.isfile(cand):
        SERVER_SCRIPT = os.path.abspath(cand)
        PROJECT_ROOT = os.path.dirname(os.path.dirname(SERVER_SCRIPT))
        break

if not SERVER_SCRIPT:
    matches = glob.glob("/content/**/webui/server.py", recursive=True) + glob.glob("**/webui/server.py", recursive=True)
    if matches:
        SERVER_SCRIPT = os.path.abspath(matches[0])
        PROJECT_ROOT = os.path.dirname(os.path.dirname(SERVER_SCRIPT))

if not SERVER_SCRIPT:
    raise FileNotFoundError("❌ Không tìm thấy file webui/server.py! Hãy chạy lại Bước 2.")

print(f"🎯 Đã định vị WebUI Server tại: {SERVER_SCRIPT}")
print(f"📂 Project Root: {PROJECT_ROOT}")
os.chdir(PROJECT_ROOT)

# 2. Download cloudflared binary nếu chưa có
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("📥 Đang tải Cloudflare Tunnel (cloudflared)...")
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

# 3. Chạy FastAPI Server ở chế độ nền
print("🚀 1. Đang khởi động ZeroTTS WebUI Server...")
server_cmd = [sys.executable, SERVER_SCRIPT, "--model", MODEL_DIR, "--host", "0.0.0.0", "--port", "7860"]
server_proc = subprocess.Popen(
    server_cmd,
    cwd=PROJECT_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

time.sleep(3)

# 4. Khởi động Cloudflare Tunnel
print("🌐 2. Đang mở đường hầm Cloudflare Tunnel...")
tunnel_cmd = ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"]
tunnel_proc = subprocess.Popen(tunnel_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

public_url = None
for _ in range(50):
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.4)
        continue
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

print("=" * 70)
if public_url:
    display(HTML(f"""
    <div style="background: linear-gradient(135deg, #1e1e2e, #2d1f47); padding: 20px; border-radius: 12px; border: 1px solid #7c3aed; margin: 15px 0;">
        <h2 style="color: #a78bfa; margin: 0 0 10px 0;">🎉 ZeroTTS Studio Đã Sẵn Sàng!</h2>
        <p style="color: #e2e8f0; font-size: 15px; margin: 0 0 15px 0;">Nhấp vào nút dưới đây để mở giao diện WebUI:</p>
        <a href="{public_url}" target="_blank" style="background: #7c3aed; color: #ffffff; padding: 12px 24px; border-radius: 8px; text-decoration: none; font-weight: bold; font-size: 16px; display: inline-block; box-shadow: 0 4px 14px rgba(124, 58, 237, 0.4);">
            🚀 Mở WebUI Studio (Public URL)
        </a>
        <p style="color: #94a3b8; font-size: 13px; margin: 12px 0 0 0;">Link: <a href="{public_url}" target="_blank" style="color: #38bdf8;">{public_url}</a></p>
    </div>
    """))
else:
    print("⚠️ Đang khởi động tunnel, vui lòng kiểm tra lại log bên dưới.")
print("=" * 70 + "\n")

# Giữ tiến trình hoạt động và hiển thị log
try:
    while True:
        line = server_proc.stdout.readline()
        if line:
            print(line, end="")
        else:
            time.sleep(0.1)
except KeyboardInterrupt:
    print("\n🛑 Đã dừng WebUI Server.")
    server_proc.terminate()
    tunnel_proc.terminate()

## ⚡ Bước 6: Chế Độ Dòng Lệnh (CLI Batch Render)
> *Sinh âm thanh hàng loạt trực tiếp trong Colab từ văn bản có gắn thẻ (`[Câu 1]`, `#[Bỏ qua]`, `[pause: 1.5s]`, `[Kết Thúc]`) mà không cần mở WebUI.*

In [ ]:
#@title Chạy Render Hàng Loạt Bằng Script { display-mode: "form" }
sample_input_text = """[Câu 1]
Xin chào các bạn. [pause: 1.5s] Đây là câu hỏi đầu tiên.

[Câu 2]
Hãy chọn đáp án đúng nhất. Thời gian suy nghĩ bắt đầu!

#[Bỏ qua]
Đoạn này nháp, hệ thống tự động bỏ qua không đọc.

[Kết Thúc]
Chúc các bạn làm bài tốt!
"""

VOICE_NAME = "nam-mien-bac" #@param ["nam-mien-bac", "nu-mien-bac", "nam-mien-nam", "nu-mien-nam", "unconditional"]
OUTPUT_PROJECT_NAME = "du_an_colab_01" #@param {type:"string"}

if PROJECT_ROOT and PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import webui.engine as engine
import os

engine.set_model(MODEL_DIR)

print(f"🎯 Đang xử lý kịch bản cho dự án: {OUTPUT_PROJECT_NAME}...")
blocks = engine.parse_input_tags(sample_input_text)

print(f"📋 Tổng số phân đoạn nhận diện: {len(blocks)}")
for b in blocks:
    status = "⏭️ BỎ QUA" if b["is_skipped"] else "✅ RENDER"
    print(f" - [{b['tag']}]: {status} | Nội dung: {b['text'][:40]}...")

# Chạy tổng hợp
voice_param = None if VOICE_NAME == "unconditional" else VOICE_NAME
mode_param = "uncond" if VOICE_NAME == "unconditional" else "voice"

print("\n🔊 Đang tiến hành sinh âm thanh và gộp file...")
out_folder = None
for event_data in engine.generate_from_parsed_blocks(
    blocks=blocks,
    voice_name=voice_param,
    mode=mode_param,
    custom_name=OUTPUT_PROJECT_NAME,
    auto_concat=True,
    merged_format="MP3"
):
    if "status" in event_data:
        print(f"   {event_data['status']}")
    if "out_folder" in event_data:
        out_folder = event_data["out_folder"]

print(f"\n🎉 Hoàn thành! Thư mục kết quả: {out_folder}")
if out_folder and os.path.exists(out_folder):
    print("📂 Danh sách file đã tạo:")
    for f in sorted(os.listdir(out_folder)):
        fpath = os.path.join(out_folder, f)
        size_kb = os.path.getsize(fpath) / 1024
        print(f"  ├── {f} ({size_kb:.1f} KB)")